<a href="https://colab.research.google.com/github/Zikai-Dou/dataflow-ai-labs/blob/main/Lab1_Profiling_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: Profiling LLM Inference Across Hardware

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Module**: 1, The Accelerator Landscape  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes


---

## Learning objectives
By the end of this lab you will be able to:
1. Build a repeatable benchmarking harness with torch.utils.benchmark for CPU and GPU inference.
2. Profile memory allocation during inference with torch.profiler and the CUDA memory stats.
3. Construct the first row of the Accelerator Landscape Map from measurements you made yourself.
4. Explain the CPU vs GPU gap in terms of SIMT parallelism, device-memory bandwidth, and precision.

## Prerequisites
Intermediate Python and PyTorch nn.Module. You have completed the Module 1 lecture on control-flow vs dataflow execution and the Accelerator Landscape Map.

## Working through this lab
1. Run the notebook cells in order.
2. Complete each learner task before opening its **Hint**.
3. If you get stuck, expand the **Hint** to view the full reference solution from the authoring notebook.
4. Run each **Self-check** cell after completing the corresponding task.



## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


## Part 0: Environment setup

We will use **Qwen 2.5 0.5B Instruct** throughout. It is small enough for Colab's free T4 GPU and representative enough of modern decoder-only LLM inference to make the measurements meaningful.

> **Why Qwen 2.5 0.5B?** It uses Grouped-Query Attention (GQA), RoPE positional embeddings, and SwiGLU; the three architectural choices that dominate modern open-weight LLMs. Whatever you learn here transfers to Llama 3, Mistral, Qwen 3, and the next model that comes out next quarter.


In [ ]:
# Install dependencies. Colab already has torch, we just need transformers and accelerate.
!pip install -q transformers==4.44.2 accelerate==0.33.0

> **Roofline & Arithmetic-Intensity Conventions (this notebook)**
> - **Precision:** FP16 (IEEE half) unless otherwise noted
> - **FLOPs counting:** 1 fused multiply-add = 2 FLOPs
> - **Memory level:** Device DRAM (HBM/GDDR), not L2 or scratchpad
> - **Byte convention:** Bytes *transferred* between DRAM and compute (reads + writes)
> - **Sparsity:** Dense (no structural sparsity) unless stated
> - **Hardware SKU:** Detected at runtime via `torch.cuda.get_device_name()`; specs from `hardware_profiles.json`

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


In [ ]:
import os
import time
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils import benchmark as torch_bench
from torch.profiler import profile, record_function, ProfilerActivity

# Make outputs reproducible
torch.manual_seed(0)

# Quick sanity: GPU runtime is required for this lab.
assert torch.cuda.is_available(), (
    "GPU runtime required. In Colab: Runtime -> Change runtime type -> T4 GPU."
)

print(f"torch       = {torch.__version__}")
print(f"CUDA        = {torch.version.cuda}")
print(f"GPU         = {torch.cuda.get_device_name(0)}")
print(f"VRAM total  = {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer + model in fp16 (inference precision on most modern GPUs)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_gpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()
model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).cpu().eval()

# Build a fixed input. Using a fixed input keeps the benchmark deterministic.
PROMPT = "Explain how a GPU differs from a dataflow accelerator in two sentences."
BATCH_SIZE = 1
SEQ_LEN = 64

inputs = tokenizer([PROMPT] * BATCH_SIZE, return_tensors="pt", padding="max_length",
                   truncation=True, max_length=SEQ_LEN)

inputs_gpu = {k: v.cuda() for k, v in inputs.items()}
inputs_cpu = {k: v.cpu() for k, v in inputs.items()}

# Print model size
n_params = sum(p.numel() for p in model_gpu.parameters())
print(f"Model: {MODEL_ID}")
print(f"Params: {n_params/1e6:.1f} M")
print(f"Input shape: {inputs['input_ids'].shape}")


## Part 1: Build a benchmarking harness

`torch.utils.benchmark.Timer` is the right tool for this: it handles GPU synchronization, warm-up, and variance estimation automatically. The naive `time.time()` around a forward pass will report *kernel launch time*, not *execution time*, because CUDA calls are asynchronous by default.

**Your task**: write a `bench_forward(model, inputs, label)` function that:
1. Warms up the model (run forward 3 times, discard results)
2. Uses `torch_bench.Timer` to measure a single forward pass
3. Returns the **median** latency in milliseconds

The `Timer` object's `.blocked_autorange()` method is what you want. It runs enough iterations to hit a stable measurement.


### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


In [ ]:
# Utility: robust timing with statistics
import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # Warmup
    for _ in range(n_warmup):
        fn()
    if sync_cuda and torch.cuda.is_available():
        torch.cuda.synchronize()

    # Measure
    times = []
    for _ in range(n_measure):
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    return {
        "median_ms": float(np.median(times)),
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "p5_ms": float(np.percentile(times, 5)),
        "p95_ms": float(np.percentile(times, 95)),
        "iqr_ms": float(np.percentile(times, 75) - np.percentile(times, 25)),
        "n": n_measure,
    }

# Example usage
if torch.cuda.is_available():
    x = torch.randn(1024, 1024, device=DEVICE)
    stats = bench_robust(lambda: torch.mm(x, x))
    print(f"Matrix multiply (1024x1024) on {DEVICE}:")
    print(f"  Median: {stats['median_ms']:.3f} ms")
    print(f"  P5-P95: [{stats['p5_ms']:.3f}, {stats['p95_ms']:.3f}] ms")
    print(f"  IQR:    {stats['iqr_ms']:.3f} ms (over {stats['n']} runs)")
else:
    print("Skipping bench_robust demo (no GPU)")


In [ ]:
    # 1.1: warm up (3 untimed forwards)
    # 1.2: Timer. We run the forward inside torch.inference_mode() so no autograd or
    #       view/version bookkeeping overhead leaks into the measurement.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
def bench_forward(model, inputs, label, min_run_time=1.0):
    """Measure median forward-pass latency (ms) using torch.utils.benchmark."""
    # 1.1: warm up (3 untimed forwards)
    with torch.inference_mode():
        for _ in range(3):
            _ = model(**inputs)
        if next(model.parameters()).is_cuda:
            torch.cuda.synchronize()

    # 1.2: Timer. We run the forward inside torch.inference_mode() so no autograd or
    #       view/version bookkeeping overhead leaks into the measurement.
    timer = torch_bench.Timer(
        stmt="with torch.inference_mode(): model(**inputs)",
        globals={"model": model, "inputs": inputs, "torch": torch},
        label=label,
        sub_label="forward",
    )

    # 1.3: blocked_autorange picks an iteration count that gives stable results.
    m = timer.blocked_autorange(min_run_time=min_run_time)
    return m.median * 1000.0  # seconds -> ms
```

</details>


In [ ]:
# --- Self-check 1 -------------------------------------------------------
# Validates your bench_forward() function on a trivial model.
class _TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(128, 128)
    def forward(self, input):
        return self.fc(input)

_check_model  = _TinyModel().cuda().eval()
_check_inputs = {"input": torch.randn(4, 128, device="cuda")}

_ms = bench_forward(_check_model, _check_inputs, "tiny-check", min_run_time=0.3)
assert _ms is not None, "bench_forward returned None; did you return the median?"
assert 0.001 < _ms < 50.0, f"Reference linear bench looks off: {_ms} ms"
print(f"[OK] bench_forward works. Reference tiny model: {_ms:.3f} ms")


## Part 2: Run inference on CPU and GPU

Now use your harness to measure the same forward pass on CPU and GPU.

**Your task**:
1. Call `bench_forward` on `model_cpu` with `inputs_cpu`.
2. Call `bench_forward` on `model_gpu` with `inputs_gpu`.
3. Compute **tokens per second** for each: `throughput = (batch_size * seq_len) / (latency_ms / 1000)`.
4. Compute the **speedup** (GPU throughput / CPU throughput).

Record the numbers in the dict `perf`.

> **What does this tokens/sec measure?** We call `model(**inputs)` once, so this is the throughput of a **single forward pass (prefill) over a fixed prompt**: how fast the model ingests the input tokens. It is *not* generated-output tokens/sec (autoregressive decode throughput), which measures how fast new tokens are produced one at a time. We look at decode throughput in a later module.


In [ ]:
# 2.1: CPU  (throughput = forward-pass / prefill tok/s over a fixed prompt,
#             NOT generated-output tok/s)
# 2.2: GPU

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
perf = {}

# 2.1: CPU  (throughput = forward-pass / prefill tok/s over a fixed prompt,
#             NOT generated-output tok/s)
cpu_ms  = bench_forward(model_cpu, inputs_cpu, "CPU (fp32)", min_run_time=0.5)
cpu_tps = (BATCH_SIZE * SEQ_LEN) / (cpu_ms / 1000.0)

# 2.2: GPU
gpu_ms  = bench_forward(model_gpu, inputs_gpu, "GPU (fp16)", min_run_time=1.0)
gpu_tps = (BATCH_SIZE * SEQ_LEN) / (gpu_ms / 1000.0)

# 2.3: speedup
speedup = gpu_tps / cpu_tps

perf["cpu_ms"]   = cpu_ms
perf["gpu_ms"]   = gpu_ms
perf["cpu_tps"]  = cpu_tps
perf["gpu_tps"]  = gpu_tps
perf["speedup"]  = speedup

print(f"CPU:    {cpu_ms:7.1f} ms/fwd   {cpu_tps:7.1f} tok/s")
print(f"GPU:    {gpu_ms:7.1f} ms/fwd   {gpu_tps:7.1f} tok/s")
print(f"Speedup (GPU / CPU): {speedup:.1f}x")
print("(tok/s = forward-pass / prefill throughput over a fixed prompt, not generated-output tok/s)")
```

</details>


In [ ]:
# --- Self-check 2 -------------------------------------------------------
# Sanity checks on your measurements. Ranges below are deliberately wide because
# Colab hardware varies. We just want to catch obvious mistakes.
assert perf["cpu_ms"] > 50,    f"CPU latency suspiciously low: {perf['cpu_ms']} ms"
assert perf["gpu_ms"] < 200,   f"GPU latency suspiciously high: {perf['gpu_ms']} ms"
assert perf["cpu_ms"] > perf["gpu_ms"], "CPU should be slower than GPU here."
assert perf["speedup"] > 1.5,  f"Expected GPU speedup > 1.5x on this workload, got {perf['speedup']:.1f}x"
print(f"[OK] Measurements look reasonable. Speedup = {perf['speedup']:.1f}x")

## Part 2b: Arithmetic Intensity Preview (Prepares for Module 6)

Now that you have measured throughput, let's compute the **arithmetic intensity** (AI) of your forward pass.
This tells you whether your workload is **compute-bound** or **memory-bound** on this GPU.

**Arithmetic Intensity** = FLOPs performed / Bytes transferred from memory

For a decoder-only transformer forward pass (rough estimate):
- FLOPs ≈ 2 × n_params × batch_size × seq_len (each parameter does one multiply-add per input token)
- Bytes ≈ n_params × 2 (read all weights in FP16 once per forward pass, at batch=1)

The **ridge point** of your GPU is: Peak TFLOPS / Peak Memory BW (TB/s).
If AI < ridge → memory-bound. If AI > ridge → compute-bound.

**Convention note:** This uses DRAM-level traffic (model weights read from device memory) and counts
a fused multiply-add as 2 FLOPs. The ridge point changes if you measure at a different memory level.


In [ ]:
# Part 2b: Compute arithmetic intensity of your forward pass
# Hardware specs lookup: normalize GPU name and use cited values.
# Source: NVIDIA T4 datasheet (2018), A100 whitepaper (2020), L4 product brief (2023)

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Part 2b: Compute arithmetic intensity of your forward pass

# Hardware specs lookup: normalize GPU name and use cited values.
# Source: NVIDIA T4 datasheet (2018), A100 whitepaper (2020), L4 product brief (2023)
HW_SPECS = {
    'T4':   {'fp16_tflops': 65.0,  'bw_tb_s': 0.320, 'source': 'NVIDIA T4 Datasheet, 2018'},
    'A100': {'fp16_tflops': 312.0, 'bw_tb_s': 2.039, 'source': 'NVIDIA A100 Whitepaper, 2020'},
    'L4':   {'fp16_tflops': 121.0, 'bw_tb_s': 0.300, 'source': 'NVIDIA L4 Product Brief, 2023'},
    'V100': {'fp16_tflops': 125.0, 'bw_tb_s': 0.900, 'source': 'NVIDIA V100 Whitepaper, 2017'},
}

def lookup_hw(gpu_name):
    for key, specs in HW_SPECS.items():
        if key in gpu_name:
            return key, specs
    return 'T4', HW_SPECS['T4']  # default fallback with warning

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'T4 (assumed)'
hw_key, hw = lookup_hw(gpu_name)
GPU_PEAK_TFLOPS_FP16 = hw['fp16_tflops']
GPU_PEAK_BW_TBS = hw['bw_tb_s']
if hw_key not in gpu_name:
    print(f"Note: GPU '{gpu_name}' not in lookup table. Using {hw_key} specs as default.")
print(f"Using {hw_key} specs: {GPU_PEAK_TFLOPS_FP16} TFLOPS FP16, {GPU_PEAK_BW_TBS} TB/s")
print(f"Source: {hw['source']}")

# 2b.1; Approximate FLOPs for one forward pass
# Convention: count fused multiply-add as 2 FLOPs (1 multiply + 1 add)
approx_flops = 2 * n_params * BATCH_SIZE * SEQ_LEN

# 2b.2; Bytes of model weights read from device memory (DRAM level)
# FP16 = 2 bytes per parameter. At batch=1, activations are negligible vs weights.
weight_bytes = n_params * 2

# 2b.3; Arithmetic intensity
ai = approx_flops / weight_bytes

# Ridge point: where compute and memory ceilings intersect
ridge_point = GPU_PEAK_TFLOPS_FP16 * 1e12 / (GPU_PEAK_BW_TBS * 1e12)  # FLOPs/byte

print(f"\nApproximate FLOPs per forward: {approx_flops/1e9:.1f} GFLOPs")
print(f"Weight bytes (FP16):           {weight_bytes/1e9:.2f} GB")
print(f"Arithmetic Intensity:          {ai:.1f} FLOPs/byte")
print(f"GPU Ridge Point ({hw_key}):       {ridge_point:.0f} FLOPs/byte")
print()
if ai < ridge_point:
    print(f"→ AI ({ai:.1f}) < Ridge ({ridge_point:.0f}): Workload is MEMORY-BOUND on {hw_key}.")
    print(f"  Tensor cores are ~{(1 - ai/ridge_point)*100:.0f}% underutilized at this batch size.")
    print(f"  This is typical for small-batch LLM inference (batch={BATCH_SIZE}, seq={SEQ_LEN}).")
    print(f"  Module 6 will explore optimization strategies for this regime.")
else:
    print(f"→ AI ({ai:.1f}) > Ridge ({ridge_point:.0f}): Workload is COMPUTE-BOUND on {hw_key}.")
    print(f"  Batch size is large enough to amortize weight reads.")

# Expected ranges (for checkpoint):
# At batch=1, seq=64: AI = batch_size * seq_len = 64 FLOPs/byte (independent of model size)
# T4 ridge = 65/0.32 ≈ 203 FLOPs/byte → memory-bound
# A100 ridge = 312/2.039 ≈ 153 FLOPs/byte → still memory-bound
assert ai == BATCH_SIZE * SEQ_LEN, f"AI should equal batch*seq={BATCH_SIZE*SEQ_LEN}, got {ai}"
print(f"\n✓ Checkpoint: AI = {ai:.0f} FLOPs/byte (= batch×seq = {BATCH_SIZE}×{SEQ_LEN})")
```

</details>


## Part 3: Profile memory allocation

Latency is half the story. On accelerators, *where and when data moves* is usually the other half. For this part you'll:

1. Measure **peak GPU memory** during a forward pass using `torch.cuda.max_memory_allocated`.
2. Use `torch.profiler` with `profile_memory=True` to capture the memory timeline of one forward pass.
3. Identify the top-3 operators by CUDA time.

This is also your first look at how a **cache-based** memory hierarchy (GPU) behaves. In later labs you'll contrast this with **software-managed** scratchpads (TPU, Spyre).


In [ ]:
# 3.1: peak memory for one forward
# 3.2: memory overhead vs model weights.
#       Compute weight bytes from the actual parameter dtypes so this stays correct

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 3.1: peak memory for one forward
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode():
    _ = model_gpu(**inputs_gpu)
torch.cuda.synchronize()
peak_bytes = torch.cuda.max_memory_allocated()

print(f"Peak GPU memory for a single forward: {peak_bytes / 1e6:.1f} MB")

# 3.2: memory overhead vs model weights.
#       Compute weight bytes from the actual parameter dtypes so this stays correct
#       even if the model is loaded in a different precision later.
weights_bytes = sum(p.numel() * p.element_size() for p in model_gpu.parameters())
mem_overhead_ratio = peak_bytes / weights_bytes
print(f"Model weights (actual dtype):         {weights_bytes / 1e6:.1f} MB")
print(f"Memory overhead ratio (peak / weights): {mem_overhead_ratio:.2f}x")
```

</details>


In [ ]:
# Use torch.profiler to capture the op-level memory timeline.
# Note: profile_memory adds overhead, we are not trying to measure latency here.
# Print top 10 operators by CUDA time.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Use torch.profiler to capture the op-level memory timeline.
# Note: profile_memory adds overhead, we are not trying to measure latency here.
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
) as prof:
    with torch.inference_mode():
        with record_function("llm_forward"):
            _ = model_gpu(**inputs_gpu)
    torch.cuda.synchronize()

# Print top 10 operators by CUDA time.
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=10,
))
```

</details>


In [ ]:
# 3.3: top 3 operators by self CUDA (device) time, excluding the record_function wrapper.
    # self device (CUDA) time in microseconds; attribute name varies across torch versions.
# Drop the "llm_forward" label. It is the record_function wrapper we added, not a real op.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 3.3: top 3 operators by self CUDA (device) time, excluding the record_function wrapper.
def _self_dev_us(e):
    # self device (CUDA) time in microseconds; attribute name varies across torch versions.
    return getattr(e, "self_device_time_total", None) or getattr(e, "self_cuda_time_total", 0.0)

# Drop the "llm_forward" label. It is the record_function wrapper we added, not a real op.
events = [e for e in prof.key_averages() if e.key != "llm_forward"]
events.sort(key=_self_dev_us, reverse=True)
top3 = [(e.key, _self_dev_us(e)) for e in events[:3]]

for name, t_us in top3:
    print(f"  {name:40s}  {t_us/1000:8.2f} ms")
```

</details>


In [ ]:
# --- Self-check 3 -------------------------------------------------------
_weights_bytes = sum(p.numel() * p.element_size() for p in model_gpu.parameters())
assert peak_bytes > _weights_bytes, "Peak memory should exceed just the model weights."
assert mem_overhead_ratio >= 1.0, f"Overhead ratio should be >= 1.0, got {mem_overhead_ratio}"
assert len(top3) == 3, "top3 must contain exactly 3 operators."
assert all(isinstance(t, tuple) and len(t) == 2 for t in top3), "top3 entries must be (name, time_us) tuples."
assert all(name != "llm_forward" for name, _ in top3), "The 'llm_forward' wrapper label must be filtered out of top3."
print("[OK] Memory profile captured, top-3 operators extracted.")

## Part 4: Build your first row of the Accelerator Landscape Map

The course's **Accelerator Landscape Map** is not vendor marketing, it's a table you build from measurement. In this lab you populate the *Compute & Memory* row for **GPU (NVIDIA T4 or whatever Colab gave you)** from the numbers you collected.

Fill in the `landscape_row` dict below using your measurements from Parts 1–3.


In [ ]:
# 4.1: Landscape Map row for the GPU.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 4.1: Landscape Map row for the GPU.
landscape_row = {
    "accelerator":          (f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"),
    "execution_model":      "SIMT / control-flow",
    "memory_hierarchy":     "L1/L2/GDDR6 global memory (hardware-managed cache)",
    "programming_model":    "CUDA / PyTorch eager",
    "forward_latency_ms":   round(perf["gpu_ms"], 2),
    "throughput_tps":       round(perf["gpu_tps"], 1),
    "peak_memory_MB":       round(peak_bytes / 1e6, 1),
    "dominant_op":          top3[0][0],
}

for k, v in landscape_row.items():
    print(f"  {k:22s}: {v}")
```

</details>


In [ ]:
# --- Self-check 4 -------------------------------------------------------
required_keys = {
    "accelerator", "execution_model", "memory_hierarchy", "programming_model",
    "forward_latency_ms", "throughput_tps", "peak_memory_MB", "dominant_op",
}
assert set(landscape_row.keys()) == required_keys, "Missing or extra keys in landscape_row."
assert all(v is not None for v in landscape_row.values()), "All landscape_row fields must be filled."
assert isinstance(landscape_row["forward_latency_ms"], (int, float))
assert isinstance(landscape_row["throughput_tps"],    (int, float))
print("[OK] Landscape Map row complete. You have contributed the first row!")

## Part 5: Analysis and Interpretation

Use the measurements from the earlier parts to explain:
1. Why GPU inference is faster than CPU inference in this workload.
2. Why peak memory usage can exceed the model-weight footprint.
3. How you expect a software-managed scratchpad dataflow accelerator to differ from the GPU memory behavior you measured.

> Write your analysis before opening the Hint below.


<details>
<summary><strong>Hint</strong></summary>

---

### 5.1: GPU speedup over CPU

A reference answer would cite:

1. **Parallelism at the thread level (SIMT).** The T4 has 2,560 CUDA cores executing in warps of 32; a modern CPU has ~4–8 cores with SIMD widths of 8–16. For GEMM-heavy workloads the raw parallel fan-out is ~100x, and the observed speedup is bounded by whatever the CPU pipeline cannot hide.
2. **Dedicated matrix engines (Tensor Cores).** The T4's Tensor Cores execute FP16 matmul at ~65 TFLOPS peak; a Xeon-class CPU delivers well under 1 TFLOPS FP32 through AVX-512. Qwen 2.5 0.5B is dominated by GEMMs in the FFN and attention, so this axis matters most.
3. **GPU device-memory bandwidth.** The T4's GDDR6 bandwidth is ~320 GB/s. A Colab CPU runtime sees maybe 20–30 GB/s of DDR4. Since LLM forward passes are increasingly memory-bound at small batch, this 10–15x bandwidth gap directly becomes a throughput gap.
4. **Numerical precision.** We ran GPU in fp16 vs CPU in fp32. Half the bytes per tensor doubles effective bandwidth and halves the compute per MAC on the Tensor Core path.

An excellent answer would also note that the **measured** speedup is lower than the peak-compute ratio would predict; that gap is where compilation, kernel launch overhead, and lack of operator fusion steal performance.

---

### 5.2: Memory overhead beyond weights

Reference answer cites the live tensors during a forward:

- **Model weights** (baseline): ~1.0 GB in fp16.
- **Activation tensors** for each layer's output, retained until consumed downstream. In pre-norm transformers, the residual stream is held live the entire depth.
- **Attention intermediates**: Q, K, V projections, Q·Kᵀ, softmax output. With 24 layers, even a short sequence accumulates intermediate tensors.
- **KV cache** (since we ran a forward pass on a 64-token input, though a prefill with no cache growth is small here).
- **PyTorch allocator fragmentation overhead**: the caching allocator over-reserves to avoid repeated cudaMalloc calls.

A good answer distinguishes *allocated* from *reserved* memory and notes that `max_memory_allocated` captures only the former.

---

### 5.3: Comparative prediction

The canonical answer: **dataflow accelerators with software-managed scratchpads typically show a *lower* peak-vs-weights ratio** for two reasons:

1. **Liveness is explicit.** The compiler schedules each tensor's scratchpad residency window. Tensors are evicted the instant the dataflow graph indicates they are no longer needed. The GPU caching allocator holds allocations well past their last use.
2. **Activations are often recomputed or streamed.** Because off-chip bandwidth is the scarce resource, dataflow compilers aggressively prefer recomputation or tile-by-tile streaming over holding activations.

A counter-prediction is also valid: **if the model doesn't fit in SRAM**, a dataflow accelerator has to spill to HBM with explicit DMAs, and the total peak allocation across HBM + SRAM can exceed the GPU equivalent. Students who flag this tradeoff get full credit.

Module 4 will revisit this with real measurements.

</details>
